# Proyecto Integrador (Grupo 4) Hito 1: Construcción del Dataset Analítico
---
## Enfoque Seleccionado: Clientes
El objetivo de este análisis es evaluar el comportamiento de compra de los usuarios, su segmentación interna y la distribución de las variables comerciales a nivel regional.

**Integrantes del grupo:**
- Ayelen Lujan Zapata Toledo
- Estephanie Vargas
- Ezequiel Martinez
- Magali Valiente
- Mateo Cruz




##  Justificación de Columnas Valoradas por Tabla
Tras una inspección directa mediante consultas SQL exploratorias, se determinó excluir los datos de carácter puramente logístico o de control interno administrativo. Se priorizó capturar únicamente las variables que describen **quién** compra, **dónde** se ubica y **cuánto** aporta al negocio.





###  1. Tabla: Clientes
*   **Columnas a Conservar:** `ID_Cliente`, `Nombre_Cliente`, `Segmento`, `Puntaje_Fidelidad`, `Canal_Preferido`.
*   **Criterio:** Son los atributos demográficos y comerciales intrínsecos del comprador. Permiten agruparlos según su tipo de consumo, lealtad y vía de contacto favorita.
*   **Columnas Descartadas:** `Codigo_Interno` (es un identificador administrativo de sistemas sin valor estadístico).

###  2. Tabla: Cliente_Ubicacion
*   **Columnas a Conservar:** `ID_Cliente`, `ID_Ubicacion`.
*   **Criterio:** Actúa estrictamente como una tabla puente relacional de normalización. Es indispensable para trazar la conexión entre el cliente y su localidad geográfica.
*   **Columnas Descartadas:** `Tipo_Direccion` (100% de registros son "Principal") y `Fecha_Actualizacion` (campo de auditoría del sistema).

###  3. Tabla: Geografia
*   **Columnas a Conservar:** `ID_Ubicacion`, `Region`, `Estado`, `Ciudad`.
*   **Criterio:** Desglosa el territorio de manera jerárquica para identificar la procedencia geográfica y estudiar el comportamiento del mercado local.
*   **Columnas Descartadas:** `Pais` (el 100% de la operación es en Chile), `Codigo_Postal`, `Zona_Logistica` y `Latitud_Ref` (variables operativas de despacho/transporte ajenas al perfil del cliente).

### 4. Tabla: Pedidos
*   **Columnas a Conservar:** `ID_Pedido`, `ID_Cliente`, `Fecha_Pedido`.
*   **Criterio:** Proporciona la trazabilidad temporal del consumo para estudiar la frecuencia de compra y la evolución cronológica del cliente.
*   **Columnas Descartadas:** `Fecha_Envio`, `Modo_Envio`, `ID_Campania` y `Observacion_Interna` (métricas de gestión de entregas y marketing secundario, sin relación directa con el perfil del cliente).
*   **`Estado_Pedido` — Nota de limitación:** esta columna **no se incorporó** al dataset final, por lo que `Monto_Total_Comprado` y `Ganancia_Total_Generada` **incluyen pedidos con cualquier estado** (incluidos cancelados y devueltos). Esto implica que un cliente con pedidos cancelados puede figurar con un monto que no representa ingreso real. Se documenta esta limitación en la sección de limpieza (Fase 3) junto al cálculo de estas columnas, y se recomienda para una futura iteración incorporar `Estado_Pedido` y recalcular los totales filtrando únicamente pedidos `'Completado'`.

###  5. Tabla: Detalle_Pedido
*   **Columnas a Conservar:** `ID_Pedido`, `Ventas`, `Ganancia`.
*   **Criterio:** Contiene el desglose financiero real de la transacción. Permite calcular el valor monetario acumulado del cliente y conocer su nivel de rentabilidad para la empresa.
*   **Columnas Descartadas:** `ID_Detalle`, `ID_Producto`, `Cantidad`, `Descuento` y `Prioridad`.

##  Importacion de dataset y librerias a utilizar




In [1]:
%load_ext sql
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%sql sqlite:///34_SuperTienda_Espanol.db


In [2]:
import pandas as pd

# Fase 1 y 2 : Exploracion y Extraccion


*  Exploración de la base de datos
*  Extracción de datos (SQL y Pandas)



###Listado de tablas disponibles en el dataset

In [3]:
%%sql
SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///34_SuperTienda_Espanol.db
Done.


name
Clientes
Productos
Geografia
Cliente_Ubicacion
Campanias
Pedidos
Detalle_Pedido
Soporte


###Seleccion de tablas y columnas a conservar

In [4]:
%%sql Clientes_Filt <<
SELECT
  id_cliente,
  Nombre_Cliente,
  Segmento,
  Puntaje_Fidelidad,
  Canal_Preferido
  FROM Clientes

 * sqlite:///34_SuperTienda_Espanol.db
Done.
Returning data to local variable Clientes_Filt


In [5]:
%%sql Cliente_Ubicacion_Filt <<
SELECT
  id_cliente,
  id_ubicacion
  FROM Cliente_Ubicacion

 * sqlite:///34_SuperTienda_Espanol.db
Done.
Returning data to local variable Cliente_Ubicacion_Filt


In [6]:
%%sql Geografia_Filt <<
SELECT
  id_ubicacion,
  Region,
  Estado,
  Ciudad
  FROM Geografia

 * sqlite:///34_SuperTienda_Espanol.db
Done.
Returning data to local variable Geografia_Filt


In [7]:
%%sql Pedidos_Filt <<
SELECT
  id_pedido,
  id_cliente,
  fecha_pedido
  FROM Pedidos

 * sqlite:///34_SuperTienda_Espanol.db
Done.
Returning data to local variable Pedidos_Filt


In [8]:
%%sql Detalle_Pedido_Filt <<
SELECT
  id_pedido,
  ventas,
  ganancia
  FROM Detalle_Pedido

 * sqlite:///34_SuperTienda_Espanol.db
Done.
Returning data to local variable Detalle_Pedido_Filt


### Creacion de variables en python - libreria pandas

In [9]:
#Declaramos Variables en Pandas
df_cliente = pd.DataFrame(Clientes_Filt)
df_cliente_ubicacion = pd.DataFrame(Cliente_Ubicacion_Filt)
df_geografia = pd.DataFrame(Geografia_Filt)
df_pedidos = pd.DataFrame(Pedidos_Filt)
df_detalle_pedido = pd.DataFrame(Detalle_Pedido_Filt)

# Fase 3: Limpieza y transformación (Python – Pandas)

## Integración de las fuentes

Unimos las cinco tablas filtradas para construir un dataset mas estructurado a nivel cliente:


*   Ventas por pedido
*   Por cliente
*   Por geografia


In [65]:
# Sumamos ventas y ganancia por pedido, ya que hay varios pedidos que tienen muchas filas

ventas_por_pedido = df_detalle_pedido.groupby('ID_Pedido').agg({
    'Ventas': 'sum',
    'Ganancia': 'sum'
})


# Renombramos los nombres de las columnas para mejorar la comprencion
ventas_por_pedido = ventas_por_pedido.rename(columns={
    'Ventas': 'Monto_Total_Pedido',
    'Ganancia': 'Ganancia_Pedido'
})


ventas_por_pedido

,Monto_Total_Pedido,Ganancia_Pedido
ID_Pedido,,
ORD-2023-00001,6427.75,2576.44
ORD-2023-00002,2059.23,644.66
ORD-2023-00003,4981.21,1444.32
ORD-2023-00004,4206.96,594.59
ORD-2023-00005,5701.27,1292.09
...,...,...
ORD-2025-01196,3833.77,644.83
ORD-2025-01197,3143.81,940.20
ORD-2025-01198,7802.40,2585.61


In [66]:
# LEFT JOIN para unir cada pedido con su monto agregado
pedidos_con_montos = df_pedidos.merge(ventas_por_pedido, on='ID_Pedido', how='left')

pedidos_con_montos

,ID_Pedido,ID_Cliente,Fecha_Pedido,Monto_Total_Pedido,Ganancia_Pedido
0,ORD-2023-00001,C0174,2025-03-31,6427.75,2576.44
1,ORD-2023-00002,C0254,2024-03-25,2059.23,644.66
2,ORD-2023-00003,C0203,2024-02-20,4981.21,1444.32
3,ORD-2023-00004,C0258,2023-10-29,4206.96,594.59
4,ORD-2023-00005,C0118,2025-03-30,5701.27,1292.09
...,...,...,...,...,...
1195,ORD-2025-01196,C0011,2023-12-16,3833.77,644.83
1196,ORD-2025-01197,C0215,2023-12-16,3143.81,940.20
1197,ORD-2025-01198,C0195,2024-06-25,7802.40,2585.61
1198,ORD-2025-01199,C0226,2024-04-12,13171.30,4012.39


⚠️ **Nota para el análisis en Power BI:**
Los montos calculados (`Monto_Total_Comprado`, `Ganancia_Total_Generada`) incluyen pedidos con
`Estado_Pedido = 'Cancelado'` y `'Devuelto'`, ya que esa columna no fue incorporada a la consulta SQL.
Para análisis de ingresos reales, considerar recalcular estos totales filtrando únicamente pedidos `'Completado'`.

In [67]:
# Agrupamos por cantidad de pedidos, monto total y ganancia total
resumen_compras_cliente = pedidos_con_montos.groupby('ID_Cliente').agg({
    'ID_Pedido': 'count',
    'Monto_Total_Pedido': 'sum',
    'Ganancia_Pedido': 'sum'
})

# Renombramos los nombres de las columnas para mejorar la comprencion
resumen_compras_cliente = resumen_compras_cliente.rename(columns={
    'ID_Pedido': 'Cantidad_Pedidos',
    'Monto_Total_Pedido': 'Monto_Total_Comprado',
    'Ganancia_Pedido': 'Ganancia_Total_Generada'
})

In [68]:
# LEFT JOIN para unir los datos del cliente con su resumen de compras
clientes_con_compras = df_cliente.merge(resumen_compras_cliente,
                                        on='ID_Cliente',
                                        how='left')

In [69]:
#Agregamos la informacion geografica de cada cliente mediante tres LEFT JOIN
# 1er L-Join Agrega el resumen de compras por cliente.
# 2do L-Join Incorpora el id de ubicacion de cada cliente.
# 3er L-Join Trae la informacion geografica asociada a esa ubicacion.

data_clientes = (
    df_cliente
    .merge(resumen_compras_cliente, on='ID_Cliente', how='left')
    .merge(df_cliente_ubicacion, on='ID_Cliente', how='left')
    .merge(df_geografia, on='ID_Ubicacion', how='left')
)


# --- Variable temporal: año de la última compra registrada del cliente ---
df_pedidos['Fecha_Pedido'] = pd.to_datetime(df_pedidos['Fecha_Pedido'], errors='coerce')

ultima_compra = df_pedidos.groupby('ID_Cliente')['Fecha_Pedido'].max().reset_index()
ultima_compra.columns = ['ID_Cliente', 'Ultima_Compra']
ultima_compra['Anio_Ultima_Compra'] = ultima_compra['Ultima_Compra'].dt.year

data_clientes = data_clientes.merge(
    ultima_compra[['ID_Cliente', 'Anio_Ultima_Compra']], on='ID_Cliente', how='left'
)

## Verificacion de tablas: Limpieza y Estanderizacion

**TABLAS A LIMPIAR**
`ventas_por_pedido ` `pedidos_con_montos` `resumen_compras_cliente` `clientes_con_compras` `data_clientes`





In [ ]:
#ventas_por_pedido.isna().sum()
#pedidos_con_montos.isna().sum()
#resumen_compras_cliente.isna().sum()
#clientes_con_compras.isna().sum()
data_clientes.isna().sum()

In [71]:
# Vetificamos si hay clientes duplicados
print("Cantidad de IDs de clientes duplicados:", data_clientes.duplicated(subset=['ID_Cliente']).sum())


Cantidad de IDs de clientes duplicados: 0


Luego de verificar cada tabla con `.isna()` y `.sum()`, se llego a la conclucion que la tabla `data_clientes` debe ser limpiada

Tambien se observo que hay dos grupos de valores faltantes:

`Canal_Preferido`: 17 nulos
`Cantidad_Pedidos`, `Monto_Total_Comprado` y `Ganancia_Total_Generada`: 11 nulos en las tres columnas, se llego a la sospecha que corresponden a los mismos clientes

Limpieza de clientes

In [ ]:
# 1. Definimos los valores de reemplazo nativos para cada columna con nulos
valores_limpieza = {
    'Cantidad_Pedidos': 0,
    'Monto_Total_Comprado': 0,
    'Ganancia_Total_Generada': 0,
    'Canal_Preferido': 'Sin Información',
    'Anio_Ultima_Compra': 0  # 0 = cliente sin compras registradas (no confundir con un año real)
}

# 2. Rellenamos masivamente todos los nulos del DataFrame final sin eliminar filas
data_clientes = data_clientes.fillna(value=valores_limpieza)
data_clientes['Anio_Ultima_Compra'] = data_clientes['Anio_Ultima_Compra'].astype(int)

# 3. Control de calidad final.
print("--- CONTROL FINAL DE NULOS POST-LIMPIEZA ---")
print(data_clientes.isna().sum())
print(f"\nTotal de clientes conservados para el análisis: {len(data_clientes)}")

In [ ]:
# variable calculada relevante

# Ticket_Promedio = 0 para clientes sin compras registradas.
# En Power BI filtrar Cantidad_Pedidos > 0 antes de calcular
# promedios globales para evitar distorsión del indicador.

# 1. División directa de columnas para calcular el Ticket Promedio
data_clientes['Ticket_Promedio'] = data_clientes['Monto_Total_Comprado'] / data_clientes['Cantidad_Pedidos']

# 2. Reemplazo nativo de NaN por 0 para los clientes registrados sin compras
data_clientes['Ticket_Promedio'] = data_clientes['Ticket_Promedio'].fillna(0)

print("'Ticket_Promedio'")
data_clientes[['ID_Cliente', 'Monto_Total_Comprado', 'Cantidad_Pedidos', 'Ticket_Promedio']].head()

#  Fase 4: Exportación del dataset
Una vez finalizado el proceso de limpieza y validación de los datos, se procede a exportar el dataset final en formato CSV.

In [ ]:
# Verificacion de la estructura final del dataset
data_clientes.info()

# Confirmamos que no existan valores nulos
print(data_clientes.isna().sum())

# Visualizacion
data_clientes.head()

## Exportacion del dataset limpio

In [ ]:
# ID_Ubicacion es un identificador interno de tabla puente, sin valor analítico en Power BI.
# La información geográfica relevante (Region, Estado, Ciudad) ya está incorporada.
data_clientes = data_clientes.drop(columns=['ID_Ubicacion'])

In [52]:
# Exportamos el dataset final en formato CSV
data_clientes.to_csv('Dataset_Clientes_Limpio.csv', index=False)

In [53]:
#Descarga del archivo (Google Colab)
from google.colab import files
files.download('Dataset_Clientes_Limpio.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# CONCLUSIONES DEL HITO 1

 **¿Qué tablas decidiste utilizar y por qué?**

- Se utilizaron las tablas de clientes, resumen de compras, cliente_ubicacion y geografia para unir la informacion personal, comercial y geografica.

- Esto nos permite tener un dataset mas completo a la hora se realizar los analisis de metricas y graficos






**¿Qué columnas seleccionaste y cuáles descartaste?**

- Se descartaron las variables que no aportaban al objetivo propuesto para el análisis (como datos logísticos) y se conservaron de forma estratégica a todos los clientes registrados, realzando sus nulos financieros a 0
- Se incorporó `Anio_Ultima_Compra` (a partir de `Fecha_Pedido`) para habilitar análisis temporales, y se descartó `ID_Ubicacion` del dataset final por ser un identificador puramente relacional sin valor analítico una vez incorporada la información geográfica (`Region`, `Estado`, `Ciudad`).
- **Limitación conocida:** `Monto_Total_Comprado` y `Ganancia_Total_Generada` incluyen pedidos cancelados y devueltos, ya que `Estado_Pedido` no fue incorporado a la consulta SQL (ver nota en Fase 3).

**¿Qué tipo de análisis permitirá tu dataset en el siguiente módulo?**

Se definieron 5 ejes de análisis, evitando preguntas redundantes que compartan la misma dimensión y métrica:

- **Ranking de clientes:** ¿qué clientes realizan más compras y cuáles generan mayores ingresos? (cantidad de pedidos vs. monto/ganancia)
- **Desempeño por zona geográfica:** ¿qué regiones/ciudades concentran más ventas y, a la vez, más clientes sin compras registradas para campañas de reactivación?
- **Canal preferido:** ¿cuál es el canal de contacto más utilizado por los clientes?
- **Ticket promedio por segmento:** ¿qué segmentos gastan más por transacción?
- **Evolución temporal:** ¿cómo evolucionó la actividad de compra de los clientes según el año de su última compra?

Cada eje corresponde a una dimensión distinta del dataset (comportamiento individual, geografía, canal, segmentación económica y tiempo), lo que permite construir un dashboard en Power BI con visuales claros y sin superposición de información.

**Auditoría y Calidad de Datos:**
- Se detectó una inconsistencia temporal en la tabla de Pedidos: los años de los códigos (ID_Pedido) no coinciden con las fechas reales (Fecha_Pedido). Por ejemplo, pedidos con código 2023 registran transacciones en 2025. Al no tener una fuente de verdad para saber cuál campo es el correcto, se tomó la decisión analítica de mantener los datos originales intactos, ya que no afectan las métricas acumuladas del Enfoque de Clientes.